# KTP Associate Skills Exercise
**Nature Broking Ltd × University of Strathclyde**

---

This exercise assesses your ability to apply NLP and textual analysis to corporate sustainability disclosures in the context of the voluntary carbon market. It is designed to take approximately **2–3 hours**.

You will:
- Characterise a corpus of 10-K filings from 62 US firms across eight sectors — Task 1
- Analyse a real carbon offset project and design a Carbon Disclosure Specificity Index — Task 2
- Conduct your own investigation into a question of your choice — Task 3
- Write a short critical synthesis

**AI assistance is permitted.** After each task, complete the AI disclosure cell: paste your prompts verbatim, summarise what the AI suggested, and explain what you accepted, modified, or rejected. The panel will use these disclosures at interview.

**Before you do anything else, run Step 0.**

In [ ]:
# ── Step 0: Environment Setup ─────────────────────────────────────────────
# Run this cell at the start of every new Colab session.
# It clones the exercise repository and loads all required libraries.
# Colab resets its environment on disconnect — always run this cell first.

import subprocess, os, sys

REPO_URL = "https://github.com/iamjamesbowden/KTP-Exercise.git"
REPO_DIR = "/content/KTP-Exercise"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repository cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True)
    print("Repository updated.")

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "scikit-learn", "nltk", "wordcloud", "-q"],
    check=True
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import nltk
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

for res in ["punkt", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger"]:
    try:
        nltk.download(res, quiet=True)
    except Exception:
        pass

CORPUS_PATH  = f"{REPO_DIR}/data/corpus.csv"
LM_PATH      = f"{REPO_DIR}/data/lm_dictionary.csv"
HARVARD_PATH = f"{REPO_DIR}/data/harvard_iv_dictionary.csv"

corpus = pd.read_csv(CORPUS_PATH)

print(f"\nCorpus loaded: {len(corpus):,} rows")
print(f"  Firms     : {corpus['firm'].nunique()}")
print(f"  Categories: {sorted(corpus['category'].unique())}")
print(f"  Sections  : {sorted(corpus['section'].unique())}")
print(f"  Years     : {sorted(corpus['year'].unique())}")

APPLICANT_NAME = input("\nEnter your full name: ").strip()
print(f"\nWelcome, {APPLICANT_NAME}. You are ready to begin.")

---

## Optional: LLM Access via OpenRouter

If you have been provided with an OpenRouter API key, you can use it to call an LLM directly within this notebook. This is entirely optional — all tasks can be completed without it.

**To set up your key:**
1. Click the **key icon** (🔑) in the left-hand sidebar of Colab to open Secrets
2. Click **Add new secret**
3. Set the name to `OPENROUTER_API_KEY` and paste your key as the value
4. Enable notebook access by toggling the switch next to the secret

**To use the key**, run the cell below. It defines a `call_llm(prompt)` helper you can call anywhere in the notebook. If no key is found, the function will raise a clear error — simply skip it and proceed without LLM access.

In [ ]:
# Optional LLM helper — requires OPENROUTER_API_KEY in Colab Secrets
# Skip this cell if you do not have a key.

import requests as _requests

def call_llm(prompt, model="anthropic/claude-3-5-haiku", max_tokens=1000):
    """
    Call an LLM via OpenRouter. Returns the response as a string.
    Logs the call so you can paste it into the AI disclosure cell.

    Parameters
    ----------
    prompt      : str   — your prompt (paste verbatim into the disclosure cell)
    model       : str   — OpenRouter model ID (default: Claude 3.5 Haiku)
    max_tokens  : int   — maximum tokens in the response
    """
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        raise RuntimeError(
            "OPENROUTER_API_KEY not found in Colab Secrets. "
            "See the setup instructions in the cell above."
        )

    resp = _requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}",
                 "Content-Type": "application/json"},
        json={"model": model,
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": max_tokens},
        timeout=60,
    )
    resp.raise_for_status()
    result = resp.json()["choices"][0]["message"]["content"]

    print("── LLM call logged ──────────────────────────────────────")
    print(f"Model : {model}")
    print(f"Prompt: {prompt[:200]}{'...' if len(prompt)>200 else ''}")
    print(f"Response ({len(result.split())} words):")
    print(result)
    print("─────────────────────────────────────────────────────────")
    return result

print("call_llm() helper ready. Use call_llm(prompt) anywhere in this notebook.")
print("Available models include: 'anthropic/claude-3-5-haiku', 'openai/gpt-4o-mini'")

---

## Research Context

### The Voluntary Carbon Market

The voluntary carbon market (VCM) enables organisations to purchase carbon credits to compensate for greenhouse gas emissions that cannot yet be eliminated through operational decarbonisation. Unlike compliance markets — such as the EU Emissions Trading System — participation in the VCM is driven by corporate sustainability commitments: net zero pledges, Science Based Targets (SBTi), and TCFD-aligned disclosure requirements, rather than by regulatory obligation.

In recent years the quality and credibility of VCM activity has come under significant scrutiny. Investigative reporting and academic research have raised concerns about offset projects that deliver substantially fewer emissions reductions than claimed, and about corporations using carbon credits as a substitute for genuine decarbonisation rather than as a last-resort mechanism for residual emissions. This has prompted the emergence of market integrity bodies — notably the Integrity Council for the Voluntary Carbon Market (ICVCM) and the Voluntary Carbon Markets Integrity Initiative (VCMI) — which publish standards for what constitutes a credible, high-quality carbon credit and a credible corporate claim.

### Nature Broking Ltd

Nature Broking is a carbon credit brokerage firm that curates and brokers premium voluntary carbon credits for corporate clients. Rather than offering transactional, volume-driven credit sales, Nature Broking positions itself as a strategic partner: conducting due diligence aligned to ICVCM standards, constructing diversified portfolios of high-quality credits, and advising clients on how to align their offset strategies with credible, science-based approaches. Their clients are corporations pursuing net zero commitments who need confidence that the credits they purchase represent genuine, permanent, and additional emissions reductions — and that their claims will withstand public and regulatory scrutiny.

### Information Asymmetry in the Voluntary Carbon Market

A central economic challenge in the VCM is information asymmetry between buyers and sellers of carbon credits. Buyers — the corporate firms in this corpus — typically cannot directly verify whether a credit represents a genuine, additional, and permanent emissions reduction. This mirrors Akerlof's (1970) "market for lemons" problem: when buyers cannot distinguish high-quality from low-quality products, adverse selection can cause markets to fail to reward quality. In the VCM, this dynamic has been documented empirically — widespread quality concerns have depressed credit prices and attracted regulatory scrutiny. Third-party standards such as the Integrity Council for the Voluntary Carbon Market (ICVCM) and the Social Carbon Standard, and specialist intermediaries such as Nature Broking, can be understood as institutional responses to this information problem — mechanisms designed to reduce asymmetry and allow credible projects to be identified and priced accordingly.

### Research Framing

A central challenge for firms operating in or adjacent to the VCM is assessing the seriousness of corporate climate commitments from public disclosure documents. Corporate filings vary enormously in how specifically and verifiably they discuss their carbon strategies — some provide detailed, quantified, and framework-referenced disclosures; others offer only vague aspirational language. Understanding this variation, and what it signals about genuine commitment versus surface-level compliance, is directly relevant to how Nature Broking identifies and advises its clients.

The question motivating this exercise is: **how specifically and credibly do US energy sector firms disclose their carbon and climate strategies in public filings, and what does the language of these disclosures reveal about the seriousness of their commitments?**

---

## About the Corpus

The corpus contains 10-K annual report filings from **62 US firms** spanning **2019–2023**, drawn from sectors known to be active in the voluntary carbon market — either as buyers of carbon credits, signatories to net zero commitments, or major TCFD reporters. Each row represents one section of one filing. Two sections are included per filing where available:

- **`item_1a`** — Risk Factors
- **`item_7`** — Management Discussion and Analysis (MD&A)

Firms are grouped into eight categories:

| Category | Firms |
|---|---|
| `oil_gas` | ExxonMobil, Chevron, ConocoPhillips, Devon Energy, EOG Resources, Hess, Kinder Morgan, Marathon Oil, Occidental Petroleum, Pioneer Natural Resources, Valero Energy, Williams Companies |
| `utility` | Ameren, American Electric Power, Dominion Energy, Duke Energy, Entergy, Exelon, NextEra Energy, PPL Corporation, Sempra Energy, Southern Company |
| `renewable` | Bloom Energy, Clearway Energy, Enphase Energy, First Solar, Plug Power, Sunnova Energy, SunPower, Sunrun |
| `airline` | Alaska Air, American Airlines, Delta Air Lines, JetBlue, Southwest Airlines, United Airlines |
| `big_tech` | Alphabet, Amazon, Apple, Meta, Microsoft, Netflix, Salesforce, Uber |
| `financial` | Bank of America, BlackRock, Citigroup, Goldman Sachs, JPMorgan Chase, Morgan Stanley, Wells Fargo |
| `consumer` | Coca-Cola, McDonald's, Nike, PepsiCo, Procter & Gamble, Starbucks, Walmart |
| `industrial` | 3M, FedEx, Honeywell, Johnson & Johnson, UPS |

**Key columns:** `firm`, `ticker`, `category`, `year`, `section`, `text`, `word_count`

Two supplementary files are available in `data/` if you wish to use them:
- `lm_dictionary.csv` — Loughran-McDonald financial sentiment dictionary
- `harvard_iv_dictionary.csv` — Harvard General Inquirer dictionary

> **Data quality note:** As with any real-world corpus drawn from regulatory filings, text quality and length vary significantly across firms and sections. Investigating these variations — and deciding how to handle them — is part of the analytical task.

---

## Task 1: Corpus Characterisation

Before analysing any corpus, it is essential to understand its structure, coverage, and vocabulary. This task asks you to produce three specific outputs. The required outputs are prescribed; how you produce them is up to you.

*Suggested time: 30–45 minutes.*

### Output 1.1 — Filing Inventory

Produce a matrix showing filing coverage across firms and years, grouped by category. Your output should make it immediately clear which firm-year combinations are present and whether both sections (Item 1A and Item 7) are available for each.

Include a summary of word count distributions by category and section. A reader should be able to identify any gaps in coverage or data quality considerations at a glance.

In [ ]:
# Output 1.1 — Filing Inventory
# Your code here


### Output 1.2 — Carbon Vocabulary Frequency

Track the frequency of carbon- and climate-related terminology in the corpus over time, broken down by firm category. The goal is to understand how the language of climate and carbon has evolved across different different types of firm in the corpus between 2019 and 2023.

A seed vocabulary is provided below. You may extend it — if you do, note your additions in the AI disclosure cell (or here, if no AI was used). Produce at least one visualisation.

**Seed vocabulary**

*Core:* `carbon offset`, `carbon credit`, `carbon neutral`, `carbon neutrality`, `net zero`, `net-zero`, `Scope 1`, `Scope 2`, `Scope 3`, `TCFD`, `SBTi`, `science-based target`, `nature-based solutions`, `sequestration`, `carbon capture`, `greenhouse gas`, `GHG`, `emissions reduction`, `climate target`, `climate commitment`, `carbon footprint`

*VCM-specific:* `Article 6`, `CORSIA`, `REDD+`, `Gold Standard`, `Verra`, `ICVCM`, `VCMI`, `insetting`, `additionality`, `permanence`, `leakage`, `biodiversity credit`, `carbon removal`, `CDR`, `verified carbon`

> **Notes on matching:** Consider using flexible pattern matching to catch plurals and hyphenated variants (e.g. `carbon offsets`, `net-zero`). Not every term will appear frequently in this corpus — the relative absence of certain VCM-specific terms is itself an analytically meaningful finding worth commenting on. Raw counts will favour longer documents; consider whether normalisation is appropriate.

> **Note on normalisation:** The `word_count` column gives the total length of each section and can serve as a denominator if you choose to normalise.

In [ ]:
# Output 1.2 — Carbon Vocabulary Frequency
# Your code here


### Output 1.3 — Disclosure Volume Trends

Examine how the volume of climate-relevant text has changed over time across the firm categories in the corpus. Are firms writing more or less about climate and carbon? Does this vary across sectors?

Design your own measure of climate-relevant disclosure volume. The `word_count` column gives total section length and may be useful as a denominator, but is not itself a measure of climate-specific content. Produce at least one visualisation.

In a short markdown cell (2–3 sentences), note what the trends suggest and whether volume alone is a meaningful indicator of the seriousness of a firm's climate engagement.

In [ ]:
# Output 1.3 — Disclosure Volume Trends
# Your code here


---

## AI Assistance Disclosure — Task 1

**Did you use an AI assistant for any part of Task 1?** *(double-click to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete a block below for each instance.*

**AI use 1**
Tool: *(e.g. ChatGPT, Claude, Copilot)*
Prompt: *(paste verbatim)*
What it suggested: *(brief summary)*
What you did with it: *(accepted / modified / rejected — and why)*

**AI use 2**
Tool:
Prompt:
What it suggested:
What you did with it:

**AI use 3**
Tool:
Prompt:
What it suggested:
What you did with it:

*(Add or remove instances as needed.)*

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach? What did you decide independently?)*

> *Your reflection here.*

---

## Task 2: Supplier-Side Analysis and Carbon Disclosure Specificity

*Suggested time: 45–60 minutes.*

### Background: The Supplier Side of the VCM

The previous task examined the demand side — what corporate buyers say about their climate strategies. This task introduces the supply side, and asks you to assess how well the two connect.

The profile below describes a real carbon project registered under the Social Carbon Standard. Read it carefully before beginning Part A.

---

**Project Profile: Boomitra Regenerative Agriculture — India**

| Attribute | Detail |
|---|---|
| **Project name** | Boomitra — Unlocking Resilience through Vital Agricultural Regeneration and Adaptation in India |
| **Registry & standard** | Social Carbon Standard |
| **Project type** | Soil carbon sequestration through regenerative agriculture |
| **Carbon mechanism** | CO₂e removed from the atmosphere and stored in agricultural soils through improved land management practices (cover cropping, rotational grazing, reduced tillage) |
| **Developer** | Boomitra — a technology-driven carbon project developer |
| **MRV approach** | Digital MRV: AI models trained on over 1 million soil samples fused with satellite imagery to quantify soil carbon sequestration at scale |
| **Scale** | Part of a 5M+ acre portfolio engaging 100,000+ smallholder farmers across multiple countries |
| **Co-benefits** | Farmer income and resilience, climate adaptation, food security |
| **Registry link** | registry.socialcarbon.org — Project ID socialcarbon-9-1695786784824x501444969412689900 |

---

### Part A — Project Analysis *(written, ~150 words)*

In the cell provided below, analyse the project profile. Address:

1. **Quality signals** — what attributes suggest this is a credible, high-quality carbon credit? Consider: the verification standard, MRV approach, co-benefits, and anything else you find meaningful.
2. **Due diligence questions** — what would a sophisticated buyer need to establish before purchasing credits from this project? Think about the standard quality criteria (additionality, permanence, leakage) and what the profile does and does not address.
3. **Economic framing** — in one or two sentences, connect your analysis to the concept of information asymmetry introduced in the briefing.

---

### Part B — Carbon Disclosure Specificity Index *(code + ~100 words)*

Design and implement a **Carbon Disclosure Specificity Index (CDSI)**: a continuous score, computed per firm-year from the corpus, measuring how specifically and credibly each filing discusses the firm's carbon strategy.

With the Boomitra profile in mind, consider: does the buyer-side language in the corpus reflect the kind of specificity that credible engagement with a project of this type would require?

There is no single correct approach. You might consider a lexicon-based approach, a rule-based system, sentence embeddings, LLM-assisted scoring, or a hybrid.

**Required outputs:**
1. A pivot table of CDSI scores (firms as rows, years as columns, grouped by category)
2. At least one visualisation
3. A written justification of your approach (100–150 words) in the cell below the code

### Part A — Your Analysis

*(Double-click to edit — ~150 words. Address quality signals, due diligence questions, and economic framing.)*

> *Your analysis here.*

In [ ]:
# Task 2 — CDSI implementation
# Your code here


In [ ]:
# Task 2 — Pivot table and visualisation
# Your code here


### Part B — Methodological Justification

*(Double-click to edit — 100–150 words.)*

Explain: the approach you took and why, the key assumptions it makes, and its main limitations.

> *Your justification here.*

---

## AI Assistance Disclosure — Task 2

**Did you use an AI assistant for any part of Task 2?** *(double-click to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete a block below for each instance.*

**AI use 1**
Tool: *(e.g. ChatGPT, Claude, Copilot)*
Prompt: *(paste verbatim)*
What it suggested: *(brief summary)*
What you did with it: *(accepted / modified / rejected — and why)*

**AI use 2**
Tool:
Prompt:
What it suggested:
What you did with it:

**AI use 3**
Tool:
Prompt:
What it suggested:
What you did with it:

*(Add or remove instances as needed.)*

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach? What did you decide independently?)*

> *Your reflection here.*

---

## Task 3: Open Investigation

*Suggested time: 45–60 minutes.*

### Instructions

Design and conduct your own investigation using the corpus loaded in Step 0. This task is an opportunity to demonstrate the analytical questions you find interesting and the methods you are most confident applying.

**Guardrails:**
- You must use the `corpus` dataframe loaded in Step 0
- Your research question must connect to at least one aspect of corporate voluntary carbon market behaviour, climate disclosure quality, or sustainability strategy
- You must produce at least one quantitative output — a table, chart, or computed statistic
- You must provide a written interpretation of your findings (100–150 words)

Strong responses will show original thinking, clear methodological reasoning, and an ability to situate findings within the VCM context introduced in the briefing. You are not expected to produce a complete research paper — focus on demonstrating how you approach and think through an analytical problem.

### Research Question and Justification

*(Double-click to edit.)*

**My research question:**
> *State your question here.*

**Why this question is relevant to Nature Broking's business context** *(2–3 sentences):*
> *Your justification here.*

In [ ]:
# Task 3 — Your code here


In [ ]:
# Task 3 — continued


In [ ]:
# Task 3 — continued


### Interpretation of Findings

*(Double-click to edit — 100–150 words.)*

In 100–150 words, summarise what your results show, what they suggest about corporate VCM behaviour in this corpus, and one logical next step.

> *Your interpretation here.*

---

## AI Assistance Disclosure — Task 3

**Did you use an AI assistant for any part of Task 3?** *(double-click to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete a block below for each instance.*

**AI use 1**
Tool: *(e.g. ChatGPT, Claude, Copilot)*
Prompt: *(paste verbatim)*
What it suggested: *(brief summary)*
What you did with it: *(accepted / modified / rejected — and why)*

**AI use 2**
Tool:
Prompt:
What it suggested:
What you did with it:

**AI use 3**
Tool:
Prompt:
What it suggested:
What you did with it:

*(Add or remove instances as needed.)*

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach? What did you decide independently?)*

> *Your reflection here.*

---

## Written Synthesis

*(Double-click to edit — 150–200 words.)*

Draw together the key patterns from Tasks 1, 2, and 3. Address:
- What the analysis reveals collectively about climate disclosure in the firms across the sectors represented in the corpus, over 2019–2023
- The main limitations of your approach across the three tasks
- How you would improve the analysis given more time or data

> *Your synthesis here.*